In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.iaa.dq_rules (
  rule_id STRING, tables STRING, column_name STRING, action STRING,
  description STRING, evidence STRING, issue STRING, approved_by STRING, approved_on DATE);

CREATE TABLE IF NOT EXISTS workspace.iaa.dq_changes (
  rule_id STRING, table_name STRING, key STRING, column_name STRING,
  raw_value STRING, clean_value STRING, run_at TIMESTAMP);

CREATE TABLE IF NOT EXISTS workspace.iaa.dq_exclusions (
  rule_id STRING, table_name STRING, key STRING, reason STRING, run_at TIMESTAMP);

CREATE TABLE IF NOT EXISTS workspace.iaa.dq_flags (
  rule_id STRING, table_name STRING, key STRING, issue STRING, run_at TIMESTAMP);

INSERT OVERWRITE workspace.iaa.dq_rules VALUES
 ('R1','policies_motor, test_motor','customer_id','correct','Left-pad to 8 characters with zeros',
  '98,110 train + 25,554 test ids short; 0% join to customers before, 100% after','#51','Cian',current_date()),
 ('R2','policies_motor, policies_buildings, policies_contents','exposure','correct','Recompute as (period_end - period_start + 1) / 365',
  '~0.6% of rows on days/360 or raw day count; 2024 rows end 30 Dec so year = 365 days','#52','Cian',current_date()),
 ('R3','policies_buildings, policies_contents, test_buildings, test_contents','sum_insured','correct','Divide by 1,000 where above 100x the training median',
  '~0.2% of rows ~1,000x typical; no rows between 10x and 100x','#53','Cian',current_date()),
 ('R4','policies_buildings, test_buildings','construction_type','correct','Map to brick / stone / timber / non_standard',
  'Mixed case; timber coded T to 2022 and W from 2023','#54','Cian',current_date()),
 ('R5','claims','*','exclude','Drop exact duplicates, keeping the standard 10-character claim_id',
  '604 pairs identical except an 11-character claim_id','#55','Cian',current_date()),
 ('R6','claims','*','flag','Loss date before the policy''s first row',
  '413 claims, mostly 2020 losses; cannot attach to a policy row','#56','Cian',current_date());

SELECT * FROM workspace.iaa.dq_rules ORDER BY rule_id;

### R1 DQ Exploration
Shows that padding is required for the joins to work correctly. 

In [0]:
%sql
SELECT length(m.customer_id)    AS id_length,
       COUNT(*)                 AS rows,
       COUNT(c_raw.customer_id) AS joins_as_is,
       COUNT(c_pad.customer_id) AS joins_after_padding
FROM workspace.iaa.policies_motor m
LEFT JOIN workspace.iaa.customers c_raw ON c_raw.customer_id = m.customer_id
LEFT JOIN workspace.iaa.customers c_pad ON c_pad.customer_id = lpad(m.customer_id, 8, '0')
GROUP BY 1
ORDER BY 1 DESC;

sample of motor pol ID issues

In [0]:
%sql
SELECT m.row_id, m.customer_id AS raw_id, lpad(m.customer_id, 8, '0') AS padded_id,
       c.date_of_birth, c.occupation, c.credit_band
FROM workspace.iaa.policies_motor m
JOIN workspace.iaa.customers c ON c.customer_id = lpad(m.customer_id, 8, '0')
WHERE length(m.customer_id) < 8
LIMIT 10;

only motor affected by short pol IDs

In [0]:
%sql
SELECT 'policies_buildings' AS t, COUNT_IF(length(customer_id) < 8) AS short_ids FROM workspace.iaa.policies_buildings
UNION ALL SELECT 'policies_contents', COUNT_IF(length(customer_id) < 8) FROM workspace.iaa.policies_contents
UNION ALL SELECT 'policies_motor',    COUNT_IF(length(customer_id) < 8) FROM workspace.iaa.policies_motor;

In [0]:
%sql
DELETE FROM workspace.iaa.dq_changes WHERE rule_id = 'R1';

INSERT INTO workspace.iaa.dq_changes
SELECT 'R1', 'policies_motor', row_id, 'customer_id', customer_id, lpad(customer_id, 8, '0'), current_timestamp()
FROM workspace.iaa.policies_motor WHERE length(customer_id) < 8
UNION ALL
SELECT 'R1', 'test_motor', row_id, 'customer_id', customer_id, lpad(customer_id, 8, '0'), current_timestamp()
FROM workspace.iaa.test_motor WHERE length(customer_id) < 8;

In [0]:
%sql
SELECT c.table_name, COUNT(*) AS cells_changed, COUNT(k.customer_id) AS join_after_fix
FROM workspace.iaa.dq_changes c
LEFT JOIN workspace.iaa.customers k ON k.customer_id = c.clean_value
WHERE c.rule_id = 'R1'
GROUP BY c.table_name;

##R2
DQ Explore

In [0]:
%sql
WITH t AS (
            SELECT 'policies_motor' AS table_name, row_id, period_start, period_end, exposure,
                   datediff(period_end, period_start) + 1 AS days FROM workspace.iaa.policies_motor
  UNION ALL SELECT 'policies_buildings', row_id, period_start, period_end, exposure,
                   datediff(period_end, period_start) + 1 FROM workspace.iaa.policies_buildings
  UNION ALL SELECT 'policies_contents',  row_id, period_start, period_end, exposure,
                   datediff(period_end, period_start) + 1 FROM workspace.iaa.policies_contents
),
classified AS (
  SELECT *, CASE
      WHEN abs(exposure - round(days / 365, 4)) < 0.0002 THEN '1. days / 365 (correct)'
      WHEN abs(exposure - round(days / 360, 4)) < 0.0002 THEN '2. days / 360'
      WHEN exposure = days                              THEN '3. raw day count'
      ELSE '4. unexplained' END AS basis
  FROM t
)
SELECT table_name, basis, COUNT(*) AS rows, MIN(exposure) AS min_exposure, MAX(exposure) AS max_exposure
FROM classified
GROUP BY ALL
ORDER BY table_name, basis;



R2 Fix

In [0]:
%sql
DELETE FROM workspace.iaa.dq_changes WHERE rule_id = 'R2';

INSERT INTO workspace.iaa.dq_changes
SELECT 'R2', t.table_name, t.row_id, 'exposure',
       CAST(t.exposure AS STRING), CAST(round(t.days / 365, 4) AS STRING), current_timestamp()
FROM (
            SELECT 'policies_motor' AS table_name, row_id, exposure, datediff(period_end, period_start) + 1 AS days FROM workspace.iaa.policies_motor
  UNION ALL SELECT 'policies_buildings', row_id, exposure, datediff(period_end, period_start) + 1 FROM workspace.iaa.policies_buildings
  UNION ALL SELECT 'policies_contents',  row_id, exposure, datediff(period_end, period_start) + 1 FROM workspace.iaa.policies_contents
) t
WHERE abs(t.exposure - round(t.days / 365, 4)) >= 0.0002;

In [0]:
%sql
SELECT table_name, COUNT(*) AS cells_changed,
       MAX(CAST(raw_value AS DOUBLE))   AS max_raw,
       MAX(CAST(clean_value AS DOUBLE)) AS max_clean
FROM workspace.iaa.dq_changes WHERE rule_id = 'R2'
GROUP BY table_name;

R3

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW r3_sum_insured AS
WITH med AS (
            SELECT 'buildings' AS line, median(sum_insured) AS train_median FROM workspace.iaa.policies_buildings
  UNION ALL SELECT 'contents',          median(sum_insured)                 FROM workspace.iaa.policies_contents
),
si AS (
            SELECT 'policies_buildings' AS table_name, 'buildings' AS line, row_id, sum_insured FROM workspace.iaa.policies_buildings
  UNION ALL SELECT 'test_buildings',    'buildings', row_id, sum_insured FROM workspace.iaa.test_buildings
  UNION ALL SELECT 'policies_contents', 'contents',  row_id, sum_insured FROM workspace.iaa.policies_contents
  UNION ALL SELECT 'test_contents',     'contents',  row_id, sum_insured FROM workspace.iaa.test_contents
)
SELECT si.*, med.train_median,
       CASE WHEN si.sum_insured > 100 * med.train_median THEN '3. over 100x median'
            WHEN si.sum_insured >  10 * med.train_median THEN '2. 10x to 100x median'
            ELSE '1. normal' END AS band
FROM si JOIN med USING (line);

In [0]:
%sql
SELECT table_name, band, COUNT(*) AS rows,
       MIN(sum_insured) AS min_si, median(sum_insured) AS median_si, MAX(sum_insured) AS max_si,
       median(sum_insured) / 1000 AS median_if_divided_by_1000
FROM r3_sum_insured
GROUP BY ALL ORDER BY table_name, band;

R3 Fix

In [0]:
%sql
DELETE FROM workspace.iaa.dq_changes WHERE rule_id = 'R3';

INSERT INTO workspace.iaa.dq_changes
SELECT 'R3', table_name, row_id, 'sum_insured',
       CAST(sum_insured AS STRING), CAST(sum_insured / 1000 AS STRING), current_timestamp()
FROM r3_sum_insured
WHERE band = '3. over 100x median';

Fix check

In [0]:
%sql
SELECT table_name, COUNT(*) AS cells_changed,
       MIN(CAST(clean_value AS DOUBLE)) AS min_clean, MAX(CAST(clean_value AS DOUBLE)) AS max_clean
FROM workspace.iaa.dq_changes WHERE rule_id = 'R3'
GROUP BY table_name ORDER BY table_name;

R4 - Construction Codes

In [0]:
%sql
SELECT * FROM (
  SELECT construction_type AS raw_code, year(period_start) AS yr FROM workspace.iaa.policies_buildings
  UNION ALL
  SELECT construction_type, year(period_start) FROM workspace.iaa.test_buildings
) PIVOT (COUNT(*) FOR yr IN (2021, 2022, 2023, 2024, 2025))
ORDER BY raw_code;

Proof that T and W are the same thing:

In [0]:
%sql
SELECT codes_seen_on_policy, COUNT(*) AS policies FROM (
  SELECT policy_id, array_join(array_sort(collect_set(upper(construction_type))), '+') AS codes_seen_on_policy
  FROM workspace.iaa.policies_buildings
  WHERE upper(construction_type) IN ('T', 'W')
  GROUP BY policy_id
) GROUP BY 1 ORDER BY 1;

Fix

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.iaa.dq_map_construction (raw_code STRING, clean_code STRING);
INSERT INTO workspace.iaa.dq_map_construction VALUES
  ('brick','brick'), ('Brick','brick'), ('BRICK','brick'),
  ('stone','stone'), ('STONE','stone'),
  ('non_standard','non_standard'), ('NON_STANDARD','non_standard'),
  ('T','timber'), ('t','timber'), ('W','timber'), ('w','timber');

Completeness check

In [0]:
%sql
SELECT b.construction_type AS unmapped_code, COUNT(*) AS rows
FROM (SELECT construction_type FROM workspace.iaa.policies_buildings
      UNION ALL SELECT construction_type FROM workspace.iaa.test_buildings) b
LEFT ANTI JOIN workspace.iaa.dq_map_construction m ON m.raw_code = b.construction_type
GROUP BY 1;

In [0]:
%sql
DELETE FROM workspace.iaa.dq_changes WHERE rule_id = 'R4';

INSERT INTO workspace.iaa.dq_changes
SELECT 'R4', b.table_name, b.row_id, 'construction_type', b.construction_type, m.clean_code, current_timestamp()
FROM (SELECT 'policies_buildings' AS table_name, row_id, construction_type FROM workspace.iaa.policies_buildings
      UNION ALL SELECT 'test_buildings', row_id, construction_type FROM workspace.iaa.test_buildings) b
JOIN workspace.iaa.dq_map_construction m ON m.raw_code = b.construction_type
WHERE b.construction_type <> m.clean_code;

Check on fix

In [0]:
%sql
SELECT table_name, raw_value, clean_value, COUNT(*) AS cells
FROM workspace.iaa.dq_changes WHERE rule_id = 'R4'
GROUP BY ALL ORDER BY table_name, clean_value, raw_value;

R5

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW r5_claims AS
SELECT *,
       length(claim_id) AS id_len,
       COUNT(*) OVER w AS copies,
       row_number() OVER (PARTITION BY policy_id, line, peril, loss_date, report_date, status,
                                       paid_to_date, incurred_to_date, recoveries
                          ORDER BY length(claim_id), claim_id) AS copy_no,
       first_value(claim_id) OVER (PARTITION BY policy_id, line, peril, loss_date, report_date, status,
                                                paid_to_date, incurred_to_date, recoveries
                                   ORDER BY length(claim_id), claim_id) AS kept_claim_id
FROM workspace.iaa.claims
WINDOW w AS (PARTITION BY policy_id, line, peril, loss_date, report_date, status,
                          paid_to_date, incurred_to_date, recoveries);

In [0]:
%sql
SELECT id_len, copies, COUNT(*) AS claims FROM r5_claims GROUP BY ALL ORDER BY id_len, copies;

In [0]:
%sql
SELECT claim_id, policy_id, line, peril, loss_date, report_date, status, paid_to_date, incurred_to_date, recoveries
FROM r5_claims WHERE copies > 1
ORDER BY policy_id, loss_date, id_len
LIMIT 12;

In [0]:
%sql
SELECT line, year(loss_date) AS loss_year, COUNT(*) AS duplicates,
       ROUND(SUM(CASE WHEN status <> 'repudiated' THEN incurred_to_date - recoveries END)) AS cost_double_counted
FROM r5_claims WHERE copy_no > 1
GROUP BY ALL ORDER BY line, loss_year;

Fix

In [0]:
%sql
DELETE FROM workspace.iaa.dq_exclusions WHERE rule_id = 'R5';

INSERT INTO workspace.iaa.dq_exclusions
SELECT 'R5', 'claims', claim_id, concat('exact duplicate of ', kept_claim_id), current_timestamp()
FROM r5_claims
WHERE copy_no > 1;

Confirm fix

In [0]:
%sql
WITH kept AS (
  SELECT c.* FROM workspace.iaa.claims c
  LEFT ANTI JOIN workspace.iaa.dq_exclusions x ON x.rule_id = 'R5' AND x.key = c.claim_id
)
SELECT (SELECT COUNT(*) FROM workspace.iaa.dq_exclusions WHERE rule_id = 'R5')                         AS excluded,
       (SELECT COUNT(*) FROM workspace.iaa.dq_exclusions WHERE rule_id = 'R5' AND length(key) = 11)    AS excluded_11_char,
       (SELECT COUNT(*) FROM kept)                                                                      AS claims_after,
       (SELECT COUNT(*) FROM (SELECT DISTINCT * EXCEPT (claim_id) FROM kept))                           AS distinct_without_id;

R6

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW r6_policy_rows AS
            SELECT 'motor' AS line, policy_id, period_start, period_end FROM workspace.iaa.policies_motor
  UNION ALL SELECT 'buildings',     policy_id, period_start, period_end FROM workspace.iaa.policies_buildings
  UNION ALL SELECT 'contents',      policy_id, period_start, period_end FROM workspace.iaa.policies_contents;

CREATE OR REPLACE TEMP VIEW r6_claims AS
WITH claims AS (
  SELECT c.* FROM workspace.iaa.claims c
  LEFT ANTI JOIN workspace.iaa.dq_exclusions x ON x.rule_id = 'R5' AND x.key = c.claim_id
),
matched AS (
  SELECT DISTINCT c.claim_id FROM claims c
  JOIN r6_policy_rows p ON p.line = c.line AND p.policy_id = c.policy_id
                       AND c.loss_date BETWEEN p.period_start AND p.period_end
),
span AS (
  SELECT line, policy_id, MIN(period_start) AS first_start, MAX(period_end) AS last_end
  FROM r6_policy_rows GROUP BY line, policy_id
),
unmatched AS (
  SELECT c.* FROM claims c LEFT ANTI JOIN matched m ON m.claim_id = c.claim_id
)
SELECT u.claim_id, u.line, u.policy_id, u.loss_date, u.status,
       u.incurred_to_date - u.recoveries AS net_cost, s.first_start, s.last_end,
       CASE WHEN s.policy_id IS NULL      THEN 'policy not in training file'
            WHEN u.loss_date < s.first_start THEN 'loss before the policy''s first row'
            WHEN u.loss_date > s.last_end    THEN 'loss after the policy''s last row'
            ELSE 'loss in a gap between rows' END AS reason
FROM unmatched u
LEFT JOIN span s ON s.line = u.line AND s.policy_id = u.policy_id;

In [0]:
%sql
SELECT line, reason, year(loss_date) AS loss_year, COUNT(*) AS claims, ROUND(SUM(net_cost)) AS cost
FROM r6_claims GROUP BY ALL ORDER BY line, reason, loss_year;

In [0]:
%sql
SELECT claim_id, line, policy_id, loss_date, first_start,
       datediff(first_start, loss_date) AS days_before_cover, status, net_cost
FROM r6_claims ORDER BY days_before_cover DESC LIMIT 10;

In [0]:
%sql
DELETE FROM workspace.iaa.dq_flags WHERE rule_id = 'R6';

INSERT INTO workspace.iaa.dq_flags
SELECT 'R6', 'claims', claim_id,
       concat(reason, ' (first row starts ', CAST(first_start AS STRING), ')'), current_timestamp()
FROM r6_claims;

In [0]:
%sql
WITH kept AS (
  SELECT c.* FROM workspace.iaa.claims c
  LEFT ANTI JOIN workspace.iaa.dq_exclusions x ON x.rule_id = 'R5' AND x.key = c.claim_id
  LEFT ANTI JOIN workspace.iaa.dq_flags      f ON f.rule_id = 'R6' AND f.key = c.claim_id
)
SELECT (SELECT COUNT(*) FROM workspace.iaa.dq_flags WHERE rule_id = 'R6') AS flagged,
       COUNT(DISTINCT k.claim_id) AS claims_attached,
       COUNT(*)                   AS claim_to_row_matches
FROM kept k
JOIN r6_policy_rows p ON p.line = k.line AND p.policy_id = k.policy_id
                     AND k.loss_date BETWEEN p.period_start AND p.period_end;

Produce updated tables and run checks on updates performed

In [0]:
%sql
-- Tables no rule touches: straight copies, so the modellers use one consistent set of _clean tables
CREATE OR REPLACE TABLE workspace.iaa.customers_clean AS SELECT * FROM workspace.iaa.customers;
CREATE OR REPLACE TABLE workspace.iaa.addresses_clean AS SELECT * FROM workspace.iaa.addresses;
CREATE OR REPLACE TABLE workspace.iaa.weather_clean   AS SELECT * FROM workspace.iaa.weather;

-- Motor: R1 customer_id, R2 exposure
CREATE OR REPLACE TABLE workspace.iaa.policies_motor_clean AS
WITH fix AS (
  SELECT key AS row_id,
         MAX(CASE WHEN column_name = 'customer_id' THEN clean_value END)                 AS customer_id,
         MAX(CASE WHEN column_name = 'exposure'    THEN CAST(clean_value AS DOUBLE) END) AS exposure
  FROM workspace.iaa.dq_changes WHERE table_name = 'policies_motor' GROUP BY key)
SELECT r.* EXCEPT (customer_id, exposure),
       COALESCE(f.customer_id, r.customer_id) AS customer_id,
       COALESCE(f.exposure,    r.exposure)    AS exposure
FROM workspace.iaa.policies_motor r LEFT JOIN fix f USING (row_id);

CREATE OR REPLACE TABLE workspace.iaa.test_motor_clean AS
WITH fix AS (
  SELECT key AS row_id,
         MAX(CASE WHEN column_name = 'customer_id' THEN clean_value END)                 AS customer_id,
         MAX(CASE WHEN column_name = 'exposure'    THEN CAST(clean_value AS DOUBLE) END) AS exposure
  FROM workspace.iaa.dq_changes WHERE table_name = 'test_motor' GROUP BY key)
SELECT r.* EXCEPT (customer_id, exposure),
       COALESCE(f.customer_id, r.customer_id) AS customer_id,
       COALESCE(f.exposure,    r.exposure)    AS exposure
FROM workspace.iaa.test_motor r LEFT JOIN fix f USING (row_id);

-- Buildings: R2 exposure, R3 sum_insured, R4 construction_type
CREATE OR REPLACE TABLE workspace.iaa.policies_buildings_clean AS
WITH fix AS (
  SELECT key AS row_id,
         MAX(CASE WHEN column_name = 'exposure'          THEN CAST(clean_value AS DOUBLE) END) AS exposure,
         MAX(CASE WHEN column_name = 'sum_insured'       THEN CAST(clean_value AS DOUBLE) END) AS sum_insured,
         MAX(CASE WHEN column_name = 'construction_type' THEN clean_value END)                 AS construction_type
  FROM workspace.iaa.dq_changes WHERE table_name = 'policies_buildings' GROUP BY key)
SELECT r.* EXCEPT (exposure, sum_insured, construction_type),
       COALESCE(f.exposure,          r.exposure)          AS exposure,
       COALESCE(f.sum_insured,       r.sum_insured)       AS sum_insured,
       COALESCE(f.construction_type, r.construction_type) AS construction_type
FROM workspace.iaa.policies_buildings r LEFT JOIN fix f USING (row_id);

CREATE OR REPLACE TABLE workspace.iaa.test_buildings_clean AS
WITH fix AS (
  SELECT key AS row_id,
         MAX(CASE WHEN column_name = 'exposure'          THEN CAST(clean_value AS DOUBLE) END) AS exposure,
         MAX(CASE WHEN column_name = 'sum_insured'       THEN CAST(clean_value AS DOUBLE) END) AS sum_insured,
         MAX(CASE WHEN column_name = 'construction_type' THEN clean_value END)                 AS construction_type
  FROM workspace.iaa.dq_changes WHERE table_name = 'test_buildings' GROUP BY key)
SELECT r.* EXCEPT (exposure, sum_insured, construction_type),
       COALESCE(f.exposure,          r.exposure)          AS exposure,
       COALESCE(f.sum_insured,       r.sum_insured)       AS sum_insured,
       COALESCE(f.construction_type, r.construction_type) AS construction_type
FROM workspace.iaa.test_buildings r LEFT JOIN fix f USING (row_id);

-- Contents: R2 exposure, R3 sum_insured
CREATE OR REPLACE TABLE workspace.iaa.policies_contents_clean AS
WITH fix AS (
  SELECT key AS row_id,
         MAX(CASE WHEN column_name = 'exposure'    THEN CAST(clean_value AS DOUBLE) END) AS exposure,
         MAX(CASE WHEN column_name = 'sum_insured' THEN CAST(clean_value AS DOUBLE) END) AS sum_insured
  FROM workspace.iaa.dq_changes WHERE table_name = 'policies_contents' GROUP BY key)
SELECT r.* EXCEPT (exposure, sum_insured),
       COALESCE(f.exposure,    r.exposure)    AS exposure,
       COALESCE(f.sum_insured, r.sum_insured) AS sum_insured
FROM workspace.iaa.policies_contents r LEFT JOIN fix f USING (row_id);

CREATE OR REPLACE TABLE workspace.iaa.test_contents_clean AS
WITH fix AS (
  SELECT key AS row_id,
         MAX(CASE WHEN column_name = 'exposure'    THEN CAST(clean_value AS DOUBLE) END) AS exposure,
         MAX(CASE WHEN column_name = 'sum_insured' THEN CAST(clean_value AS DOUBLE) END) AS sum_insured
  FROM workspace.iaa.dq_changes WHERE table_name = 'test_contents' GROUP BY key)
SELECT r.* EXCEPT (exposure, sum_insured),
       COALESCE(f.exposure,    r.exposure)    AS exposure,
       COALESCE(f.sum_insured, r.sum_insured) AS sum_insured
FROM workspace.iaa.test_contents r LEFT JOIN fix f USING (row_id);

-- Claims: R5 duplicates removed. The R6 claims stay in; they simply attach to no policy row.
CREATE OR REPLACE TABLE workspace.iaa.claims_clean AS
SELECT c.* FROM workspace.iaa.claims c
LEFT ANTI JOIN workspace.iaa.dq_exclusions x ON x.table_name = 'claims' AND x.key = c.claim_id;

In [0]:
from pyspark.sql import functions as F
import pandas as pd

S = "workspace.iaa"
logged = (spark.table(f"{S}.dq_changes").groupBy("table_name", "column_name").count()
          .toPandas().set_index(["table_name", "column_name"])["count"].to_dict())
excluded = (spark.table(f"{S}.dq_exclusions").groupBy("table_name").count()
            .toPandas().set_index("table_name")["count"].to_dict())

checks = []
for t, key in [("policies_motor", "row_id"), ("test_motor", "row_id"),
               ("policies_buildings", "row_id"), ("test_buildings", "row_id"),
               ("policies_contents", "row_id"), ("test_contents", "row_id"),
               ("claims", "claim_id"), ("customers", "customer_id"), ("addresses", "address_id")]:
    raw, clean = spark.table(f"{S}.{t}"), spark.table(f"{S}.{t}_clean")
    checks.append((t, "(rows)", raw.count() - excluded.get(t, 0), clean.count()))
    cols = [c for c in raw.columns if c != key]
    diffs = (raw.alias("r").join(clean.alias("c"), key)
             .select([F.sum((~F.col(f"r.{c}").eqNullSafe(F.col(f"c.{c}"))).cast("int")).alias(c) for c in cols])
             .first().asDict())
    checks += [(t, c, logged.get((t, c), 0), diffs[c] or 0) for c in cols]

w_raw, w_clean = spark.table(f"{S}.weather"), spark.table(f"{S}.weather_clean")
checks.append(("weather", "(identical)", 0, w_raw.exceptAll(w_clean).count() + w_clean.exceptAll(w_raw).count()))

rec = pd.DataFrame(checks, columns=["table", "column", "expected", "actual"])
rec["status"] = (rec["expected"] == rec["actual"]).map({True: "PASS", False: "FAIL"})
display(rec[(rec["expected"] != 0) | (rec["actual"] != 0) | (rec["status"] == "FAIL")])
assert (rec["status"] == "PASS").all(), "Reconciliation FAILED: see the rows above"
print(f"Reconciliation passed: {len(rec)} checks; every changed cell is logged and every row is accounted for.")

In [0]:
%sql
COMMENT ON TABLE workspace.iaa.policies_motor_clean     IS 'Week 1 motor training rows, cleaned (R1, R2). See docs/DATA_CONTRACT.md';
COMMENT ON TABLE workspace.iaa.test_motor_clean         IS 'Week 1 motor 2025 rows to predict, cleaned (R1). See docs/DATA_CONTRACT.md';
COMMENT ON TABLE workspace.iaa.policies_buildings_clean IS 'Week 1 buildings training rows, cleaned (R2, R3, R4). See docs/DATA_CONTRACT.md';
COMMENT ON TABLE workspace.iaa.test_buildings_clean     IS 'Week 1 buildings 2025 rows to predict, cleaned (R3, R4). See docs/DATA_CONTRACT.md';
COMMENT ON TABLE workspace.iaa.policies_contents_clean  IS 'Week 1 contents training rows, cleaned (R2, R3). See docs/DATA_CONTRACT.md';
COMMENT ON TABLE workspace.iaa.test_contents_clean      IS 'Week 1 contents 2025 rows to predict, cleaned (R3). See docs/DATA_CONTRACT.md';
COMMENT ON TABLE workspace.iaa.claims_clean             IS 'Week 1 claims with 604 duplicates removed (R5); 413 flagged in dq_flags (R6). See docs/DATA_CONTRACT.md';